# Layer Symmetry Tests: Left/Right Action — Invariance & Equivariance

## Valid tests per layer

| Layer | Left crystal-group (s⊗q) | Right crystal-group (q⊗s) | SO(3) on features |
|---|---|---|---|
| `LocalIsoCrystalEncoder` | **invariance**: enc(s⊗q)=enc(q) | **right-equivariance**: enc(q⊗s)=enc(q)@D(s) | ❌ N/A (contradicts left-invariance) |
| `EquivariantSpatialConv` | ❌ N/A (no quaternion input) | ❌ N/A | **equivariance**: f(D·x)=D·f(x) |
| `EquivariantTransposeConv` | ❌ N/A | ❌ N/A | **equivariance**: f(D·x)=D·f(x) |
| `AttentionBlock` | ❌ N/A | ❌ N/A | **equivariance** of delta + **invariance** of attn weights |

### Why SO(3) left-equivariance is N/A for the encoder
The encoder satisfies `enc(s⊗q) = enc(q)` (left-O-invariant). If it were ALSO left-SO(3)-equivariant,
we'd need `D(s)=I` for all `s ∈ O`. But `1x4e` restricted to O is not trivial (l=4 decomposes as A1+E+T1+T2).
The two properties are contradictory — left-O-invariance wins by construction.

### Right-action: equivariance, not invariance
From the Haar-integral structure of local-iso: `enc(q⊗g) = enc(q) @ D⁴(g)` (right-multiply features by D).  
This means `‖enc(q_lr⊗s) - enc(q_hr⊗s)‖ = ‖enc(q_lr) - enc(q_hr)‖` (D is unitary → MSE preserved).

**e3nn tools used:** `o3.rand_matrix()`, `irreps.D_from_matrix(R)`, `quat_to_matrix` for sym_ops

In [1]:
from __future__ import annotations
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
sys.path.insert(0, os.path.abspath(os.path.dirname('__file__')))

import torch
import torch.nn.functional as F
from e3nn import o3
from e3nn.o3 import Irreps

from _helpers import (
    normalize_quaternions, quat_mul, quat_to_matrix,
    rand_quaternions, rotate_features, rel_error,
    report, section, summary,
)
from models.SR_double_conv_SRattn_a1 import (
    LocalIsoCrystalEncoder,
    EquivariantSpatialConv,
    EquivariantTransposeConv,
    AttentionBlock,
)

In [2]:
# ── config ────────────────────────────────────────────────────────────────────
CRYSTAL    = 'fcc'    # change to 'hcp'
H, W       = 8, 8
BLOCK_SIZE = 4
UPSAMPLE   = 4
N_TRIALS   = 16
N_QUATS    = 64
TOL        = 1e-4
DEVICE     = torch.device('cpu')
SEED       = 42

enc      = LocalIsoCrystalEncoder(crystal=CRYSTAL, dtype=torch.float32, device=DEVICE).eval()
irr_a1   = enc.irreps_a1
irr_full = enc.irreps_full
sym_ops  = enc.sym_ops      # (n_ops, 4) — crystal group quaternions
sym_inv  = enc.sym_ops_inv  # conjugates
n_ops    = sym_ops.shape[0]

print(f'Crystal    : {CRYSTAL.upper()}')
print(f'irreps_a1  : {irr_a1}   dim={irr_a1.dim}')
print(f'irreps_full: {irr_full}  dim={irr_full.dim}')
print(f'|S|        : {n_ops}')

Crystal    : FCC
irreps_a1  : 1x4e   dim=9
irreps_full: 1x2e+1x4e  dim=14
|S|        : 24


## Helpers

In [3]:
# ── SO(3) equivariance on feature vectors ─────────────────────────────────────
def so3_equivariance_errors(fn, x, irreps_in, irreps_out, n_trials, seed):
    """
    e3nn test: f( x @ D_in(R).T ) ≈ f(x) @ D_out(R).T   for random R ∈ SO(3).
    Both sides rotate by the SAME R; uses o3.rand_matrix() and D_from_matrix.
    Only valid for layers that take feature vectors as input (not quaternions).
    """
    with torch.no_grad():
        y_ref = fn(x)
    errors = []
    torch.manual_seed(seed)
    for _ in range(n_trials):
        R    = o3.rand_matrix(dtype=torch.float32).to(DEVICE)
        D_in = irreps_in.D_from_matrix(R).to(DEVICE)
        D_out= irreps_out.D_from_matrix(R).to(DEVICE)
        with torch.no_grad():
            err = rel_error(fn(rotate_features(x, D_in)),   # f(D·x)
                            rotate_features(y_ref, D_out))   # D·f(x)
        errors.append(err)
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max())


# ── Crystal-group equivariance on feature vectors (O ⊂ SO(3)) ─────────────────
def crystal_so3_equivariance_errors(fn, x, sym_ops, irreps, n_ops):
    """
    Tests f(D(s)·x) = D(s)·f(x) for each s ∈ crystal group.
    Uses the sym_ops quaternions as concrete SO(3) elements via D_from_matrix.
    Since O ⊂ SO(3) and layers are SO(3)-equivariant, this must PASS.
    """
    with torch.no_grad():
        y_ref = fn(x)
    errors = []
    for i in range(n_ops):
        R_s = quat_to_matrix(sym_ops[i])
        D_s = irreps.D_from_matrix(R_s).to(DEVICE)
        with torch.no_grad():
            err = rel_error(fn(rotate_features(x, D_s)),
                            rotate_features(y_ref, D_s))
        errors.append(err)
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max()), errors


# ── Encoder: left crystal-group invariance ─────────────────────────────────────
def encoder_left_invariance_errors(encode_fn, q, sym_ops):
    """
    enc(s ⊗ q) ≈ enc(q)  for all s ∈ crystal group.
    Left-action on orientations; features must be IDENTICAL.
    """
    with torch.no_grad():
        feat_ref = encode_fn(q)
    errors = []
    for i in range(sym_ops.shape[0]):
        s     = sym_ops[i].unsqueeze(0).expand(q.shape[0], -1)
        q_sym = normalize_quaternions(quat_mul(s, q))   # s ⊗ q
        with torch.no_grad():
            errors.append(rel_error(encode_fn(q_sym), feat_ref))
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max()), errors


# ── Encoder: right crystal-group equivariance ──────────────────────────────────
def encoder_right_equivariance_errors(encode_fn, q, sym_ops, irreps_out):
    """
    enc(q ⊗ s) ≈ enc(q) @ D(s)   for all s ∈ crystal group.

    RIGHT-multiplication of feature row-vector by D(s), NOT rotate_features.
    Derivation from Haar-integral structure:
        enc(q⊗g) = ∫_O D⁴(s·q·g) ds = [∫_O D⁴(sq) ds] · D⁴(g) = enc(q) @ D⁴(g)

    Note: enc(q⊗s) ≠ enc(q)  (not invariant) but the MSE loss IS invariant because
    ‖enc(q_lr)@D − enc(q_hr)@D‖ = ‖enc(q_lr) − enc(q_hr)‖  (D unitary).
    """
    with torch.no_grad():
        feat_ref = encode_fn(q)   # (N, C)
    errors = []
    for i in range(sym_ops.shape[0]):
        s     = sym_ops[i].unsqueeze(0).expand(q.shape[0], -1)
        R_s   = quat_to_matrix(sym_ops[i])
        D_s   = irreps_out.D_from_matrix(R_s).to(DEVICE)   # (C, C)
        q_sym = normalize_quaternions(quat_mul(q, s))       # q ⊗ s
        with torch.no_grad():
            feat_sym  = encode_fn(q_sym)        # enc(q⊗s)
            feat_pred = feat_ref @ D_s          # enc(q) @ D(s)  ← RIGHT-multiply
        errors.append(rel_error(feat_sym, feat_pred))
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max()), errors


# ── AttentionBlock: attention weight invariance ────────────────────────────────
def attn_weight_invariance_errors(block, x, irreps, d_block, n_trials, seed):
    """
    attn(D·x) ≈ attn(x).  D orthogonal → (Df̂_i)·(Df̂_j) = f̂_i·f̂_j → scores unchanged.
    """
    bh = bw = BLOCK_SIZE
    B, _, C = x.shape
    num_bh = H//bh; num_bw = W//bw; Nb = bh*bw; Bb = B*num_bh*num_bw

    def _attn(feat):
        fb = (feat.reshape(B,num_bh,bh,num_bw,bw,C)
                  .permute(0,1,3,2,4,5).reshape(Bb,Nb,C))
        fn = F.normalize(fb, dim=-1)
        sc = torch.exp(block.log_s)*torch.bmm(fn, fn.transpose(-2,-1))
        pb = block.pos_bias(d_block.unsqueeze(-1)).squeeze(-1)
        return torch.softmax((sc+pb.unsqueeze(0)).float(), dim=-1)

    with torch.no_grad():
        a_ref = _attn(x)
    errors = []
    torch.manual_seed(seed)
    for _ in range(n_trials):
        R = o3.rand_matrix(dtype=torch.float32).to(DEVICE)
        D = irreps.D_from_matrix(R).to(DEVICE)
        with torch.no_grad():
            errors.append(rel_error(_attn(rotate_features(x, D)), a_ref))
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max())


print('Helpers defined.')


# ── Crystal-group LEFT-invariance on feature vectors ─────────────────────────
def feature_left_invariance_errors(fn, x, sym_ops, irreps_in, n_ops):
    """
    Tests f(D(s)·x) = f(x) for each s in the crystal group.
    Equivariant layers satisfy f(D·x)=D·f(x), NOT f(D·x)=f(x), so this FAILS.
    The only way it passes is if D(s) acts trivially (scalar irreps only).
    Contrast with encoder left-invariance which operates on quaternion inputs.
    """
    with torch.no_grad():
        y_ref = fn(x)
    errors = []
    for i in range(n_ops):
        R_s   = quat_to_matrix(sym_ops[i])
        D_s   = irreps_in.D_from_matrix(R_s).to(DEVICE)
        x_rot = rotate_features(x, D_s)   # D(s)·x
        with torch.no_grad():
            y_rot = fn(x_rot)
        errors.append(rel_error(y_rot, y_ref))
    t = torch.tensor(errors)
    return float(t.mean()), float(t.max()), errors


Helpers defined.


---
## 1. LocalIsoCrystalEncoder

| Test | Expected | Reason |
|---|---|---|
| Left crystal invariance `enc(s⊗q)=enc(q)` | ✅ PASS | Fundamental local-iso property |
| Right crystal equivariance `enc(q⊗s)=enc(q)@D(s)` | ✅ PASS | Haar-integral → right-multiply by D |
| SO(3) left-equivariance `enc(g⊗q)=D(g)·enc(q)` | ❌ N/A — **not tested** | Contradicts left-O-invariance |

In [4]:
# ── Passive→Active convention check ──────────────────────────────────────────
# All quaternions entering the encoder are treated as PASSIVE (MTEX/EBSD convention).
# The encoder internally calls _quat_conjugate(q_passive) → q_active before embedding.
# We verify this by comparing two paths that must give identical features:
#   Path A: encoder.forward_a1(q_passive)           — production path (conjugates inside)
#   Path B: embedding.forward_from_quaternions(q_active)  — manual active path (no conjugation)

from _helpers import quat_conjugate

q_passive_check = rand_quaternions(N_QUATS, SEED, DEVICE)
q_active_check  = quat_conjugate(q_passive_check)   # passive → active: negate xyz

with torch.no_grad():
    # Path A: encoder receives passive, conjugates internally
    feat_A = enc.embedding.forward_irreps_passive(q_passive_check, active_only=True)
    # Path B: manually converted to active, uses active path directly
    feat_B = enc.embedding.forward_from_quaternions(q_active_check, active_only=True)

max_err = float((feat_A - feat_B).abs().max())
status  = 'PASS' if max_err < 1e-5 else 'FAIL'
print(f'[{status}] Passive→Active convention check  max_abs_err={max_err:.2e}')
print(f'        q_passive → conj → q_active → R → irreps')
print(f'        Path A (enc.forward_a1 via forward_irreps_passive) == Path B (forward_from_quaternions)')
assert max_err < 1e-5, f'Convention mismatch! max_err={max_err:.2e}'


[PASS] Passive→Active convention check  max_abs_err=0.00e+00
        q_passive → conj → q_active → R → irreps
        Path A (enc.forward_a1 via forward_irreps_passive) == Path B (forward_from_quaternions)


In [5]:
results = []
section('LocalIsoCrystalEncoder')
q_passive = rand_quaternions(N_QUATS, SEED, DEVICE)

# ── Left crystal invariance ────────────────────────────────────────────────────
print('  1a. Left crystal invariance: enc(s⊗q_passive) = enc(q_passive)')
for fn_name, fn in [('forward_a1  ', enc.forward_a1),
                    ('forward_full', enc.forward_full)]:
    mean_e, max_e, _ = encoder_left_invariance_errors(fn, q_passive, sym_ops)
    ok = report(f'Encoder {fn_name} LEFT-invariance  enc(s⊗q_passive)=enc(q_passive)',
                max_e, TOL, extra=f'mean={mean_e:.2e}  over {n_ops} ops')
    results.append(ok)

# ── Right crystal equivariance ─────────────────────────────────────────────────
print('\n  1b. Right crystal equivariance: enc(q_passive⊗s) = enc(q_passive) @ D(s)')
print('      [enc(q_passive⊗s) ≠ enc(q_passive): not invariant, but MSE-loss is preserved because D is unitary]')
for fn_name, fn, irr_out in [
        ('forward_a1  ', enc.forward_a1,   irr_a1),
        ('forward_full', enc.forward_full, irr_full)]:
    mean_e, max_e, _ = encoder_right_equivariance_errors(fn, q_passive, sym_ops, irr_out)
    ok = report(f'Encoder {fn_name} RIGHT-equivariance enc(q_passive⊗s)=enc(q_passive)@D(s)',
                max_e, TOL, extra=f'mean={mean_e:.2e}  over {n_ops} ops')
    results.append(ok)

# ── SO(3) left-equivariance: N/A (shown as informational FAIL) ─────────────────
print('\n  1c. SO(3) left-equivariance: N/A — shown only to confirm expected FAIL')
print('      enc(g⊗q_passive) ≠ D(g)·enc(q_passive) because left-O-invariance is contradictory with non-trivial D(s) for s∈O')
torch.manual_seed(SEED)
with torch.no_grad():
    feat_q = enc.forward_a1(q_passive)
so3_errs = []
for _ in range(N_TRIALS):
    # generate random g ∈ SO(3) as a quaternion
    g_q = normalize_quaternions(torch.randn(1, 4))
    R_g = quat_to_matrix(g_q[0])
    D_g = irr_a1.D_from_matrix(R_g).to(DEVICE)
    g_exp = g_q.expand(N_QUATS, -1)
    q_rot = normalize_quaternions(quat_mul(g_exp, q_passive))
    with torch.no_grad():
        so3_errs.append(rel_error(enc.forward_a1(q_rot), rotate_features(feat_q, D_g)))
t = torch.tensor(so3_errs)
report('Encoder SO(3) left-equivariance [INFORMATIONAL — expected FAIL]',
       float(t.max()), TOL, extra=f'mean={float(t.mean()):.2e}  (FAIL = correct)')

summary(results)


# ── 1d. A1 subspace check: D(s)·enc(q) = enc(q) for all s ∈ S ───────────────
# If left-invariance holds, Haar structure gives D(s)·enc(q) = enc(s⊗q) = enc(q).
# So encoder features must be FIXED POINTS of D(s) — they live in the A1 subspace.
# This is a prerequisite for the spatial layer left-invariance tests below.
print('\n  1d. A1 subspace: D(s)·enc(q_passive) = enc(q_passive)')
for fn_name, fn, irr_out in [
        ('forward_a1  ', enc.forward_a1,   irr_a1),
        ('forward_full', enc.forward_full, irr_full)]:
    with torch.no_grad():
        feat = fn(q_passive)   # (N, C)
    a1_errors = []
    for i in range(n_ops):
        R_s = quat_to_matrix(sym_ops[i])
        D_s = irr_out.D_from_matrix(R_s).to(DEVICE)
        feat_rotated = rotate_features(feat, D_s)   # D(s)·feat
        a1_errors.append(rel_error(feat_rotated, feat))
    t = torch.tensor(a1_errors)
    ok = report(f'Encoder {fn_name} A1 subspace: D(s)·enc(q)=enc(q)',
                float(t.max()), TOL, extra=f'mean={float(t.mean()):.2e}  over {n_ops} ops')
    results.append(ok)



──────────────────────────────────────────────────────────────────────
  LocalIsoCrystalEncoder
──────────────────────────────────────────────────────────────────────
  1a. Left crystal invariance: enc(s⊗q_passive) = enc(q_passive)
  [PASS] Encoder forward_a1   LEFT-invariance  enc(s⊗q_passive)=enc(q_passive)  rel=5.70e-07  tol=1e-04  (mean=4.93e-07  over 24 ops)
  [PASS] Encoder forward_full LEFT-invariance  enc(s⊗q_passive)=enc(q_passive)  rel=7.53e-07  tol=1e-04  (mean=6.52e-07  over 24 ops)

  1b. Right crystal equivariance: enc(q_passive⊗s) = enc(q_passive) @ D(s)
      [enc(q_passive⊗s) ≠ enc(q_passive): not invariant, but MSE-loss is preserved because D is unitary]
  [PASS] Encoder forward_a1   RIGHT-equivariance enc(q_passive⊗s)=enc(q_passive)@D(s)  rel=6.55e-06  tol=1e-04  (mean=3.04e-06  over 24 ops)
  [PASS] Encoder forward_full RIGHT-equivariance enc(q_passive⊗s)=enc(q_passive)@D(s)  rel=6.56e-06  tol=1e-04  (mean=3.08e-06  over 24 ops)

  1c. SO(3) left-equivariance: N/A 

---
## 2. EquivariantSpatialConv — SO(3) equivariance on feature vectors

Spatial layers take **feature vectors**, not quaternions. Crystal group acts on them via Wigner-D.

| Test | Expected |
|---|---|
| SO(3) equivariance (random R) | ✅ PASS — e3nn FCTP guarantee |
| Crystal-group equivariance (all 24/12 ops) | ✅ PASS — O ⊂ SO(3) |
| Trivial action `D(s)·x = x` (informational) | ❌ FAIL for `1x4e` — l=4 is not trivial under O |

In [6]:
results = []
section('EquivariantSpatialConv')

torch.manual_seed(SEED)
x_a1 = torch.randn(1, H*W, irr_a1.dim, dtype=torch.float32, device=DEVICE)

for res in (False, True):
    torch.manual_seed(SEED + int(res))
    conv = EquivariantSpatialConv(
        kernel_size=3, irreps_in=irr_a1, irreps_out=irr_a1, use_residual=res
    ).to(DEVICE).eval()
    fn = lambda x, _c=conv: _c(x, (H, W))

    # SO(3) equivariance — random rotations via o3.rand_matrix()
    mean_e, max_e = so3_equivariance_errors(fn, x_a1, irr_a1, irr_a1, N_TRIALS, SEED)
    results.append(report(f'SpatialConv(res={res}) SO(3) equivariance',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

    # Crystal-group equivariance — all sym_ops as concrete SO(3) elements
    mean_e, max_e, _ = crystal_so3_equivariance_errors(fn, x_a1, sym_ops, irr_a1, n_ops)
    results.append(report(f'SpatialConv(res={res}) crystal-group equivariance ({n_ops} ops)',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

    # Trivial action check (informational — expected FAIL for non-scalar irreps)
    triv = [rel_error(rotate_features(x_a1, irr_a1.D_from_matrix(quat_to_matrix(sym_ops[i])).to(DEVICE)), x_a1)
            for i in range(n_ops)]
    t = torch.tensor(triv)
    report(f'SpatialConv(res={res}) trivial action D(s)·x=x [INFO: FAIL expected for {irr_a1}]',
           float(t.max()), TOL, extra=f'mean={float(t.mean()):.2e}')

summary(results)


──────────────────────────────────────────────────────────────────────
  EquivariantSpatialConv
──────────────────────────────────────────────────────────────────────
  [PASS] SpatialConv(res=False) SO(3) equivariance                rel=1.40e-05  tol=1e-04  (mean=5.53e-06)
  [PASS] SpatialConv(res=False) crystal-group equivariance (24 ops)  rel=9.77e-06  tol=1e-04  (mean=3.75e-06)
  [FAIL] SpatialConv(res=False) trivial action D(s)·x=x [INFO: FAIL expected for 1x4e]  rel=1.44e+00  tol=1e-04  (mean=1.31e+00)
  [PASS] SpatialConv(res=True) SO(3) equivariance                 rel=3.77e-06  tol=1e-04  (mean=1.49e-06)
  [PASS] SpatialConv(res=True) crystal-group equivariance (24 ops)  rel=2.63e-06  tol=1e-04  (mean=1.01e-06)
  [FAIL] SpatialConv(res=True) trivial action D(s)·x=x [INFO: FAIL expected for 1x4e]  rel=1.44e+00  tol=1e-04  (mean=1.31e+00)

  [PASS]  4/4 tests passed



In [7]:
# ── Left crystal-group invariance with ENCODER features ──────────────────────
# Encoder features lie in the A1 subspace: D(s)·feat = feat (verified above).
# Equivariant layer: f(D(s)·feat) = D(s)·f(feat) = f(feat)  ← PASS expected.
# (Random vectors would fail — this test is only meaningful with real enc features.)

section('SpatialConv — Left crystal-group invariance on encoder features')

q_spat = rand_quaternions(H * W, SEED, DEVICE)
with torch.no_grad():
    x_enc = enc.forward_a1(q_spat).unsqueeze(0)   # (1, H*W, C)

inv_results = []
for res in (False, True):
    torch.manual_seed(SEED + int(res))
    conv = EquivariantSpatialConv(
        kernel_size=3, irreps_in=irr_a1, irreps_out=irr_a1, use_residual=res
    ).to(DEVICE).eval()
    fn = lambda x, _c=conv: _c(x, (H, W))

    mean_e, max_e, _ = feature_left_invariance_errors(fn, x_enc, sym_ops, irr_a1, n_ops)
    ok = report(
        f'SpatialConv(res={res}) left-inv on enc features: f(D(s)·feat)=f(feat)',
        max_e, TOL, extra=f'mean={mean_e:.2e}  over {n_ops} ops',
    )
    inv_results.append(ok)

summary(inv_results)



──────────────────────────────────────────────────────────────────────
  SpatialConv — Left crystal-group invariance on encoder features
──────────────────────────────────────────────────────────────────────
  [FAIL] SpatialConv(res=False) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=1.47e+00  tol=1e-04  (mean=1.34e+00  over 24 ops)
  [FAIL] SpatialConv(res=True) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=1.46e+00  tol=1e-04  (mean=1.32e+00  over 24 ops)

  [FAIL]  0/2 tests passed



---
## 3. EquivariantTransposeConv — SO(3) equivariance + shape check

In [8]:
results = []
section('EquivariantTransposeConv')

torch.manual_seed(SEED)
x_a1 = torch.randn(1, H*W, irr_a1.dim, dtype=torch.float32, device=DEVICE)

for res in (False, True):
    torch.manual_seed(SEED + int(res))
    tc = EquivariantTransposeConv(
        kernel_size=3, upsample_factor=UPSAMPLE,
        use_residual=res, irreps_in=irr_a1, irreps_out=irr_a1
    ).to(DEVICE).eval()
    fn = lambda x, _tc=tc: _tc(x, (H, W))[0]

    with torch.no_grad():
        _, (Hr, Wr) = tc(x_a1, (H, W))
    ok = (Hr == H*UPSAMPLE) and (Wr == W*UPSAMPLE)
    results.append(ok)
    print(f'  [{"PASS" if ok else "FAIL"}] TransposeConv(res={res}) shape ({Hr},{Wr}) == ({H*UPSAMPLE},{W*UPSAMPLE})')

    mean_e, max_e = so3_equivariance_errors(fn, x_a1, irr_a1, irr_a1, N_TRIALS, SEED)
    results.append(report(f'TransposeConv(res={res}) SO(3) equivariance',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

    mean_e, max_e, _ = crystal_so3_equivariance_errors(fn, x_a1, sym_ops, irr_a1, n_ops)
    results.append(report(f'TransposeConv(res={res}) crystal-group equivariance ({n_ops} ops)',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

summary(results)


──────────────────────────────────────────────────────────────────────
  EquivariantTransposeConv
──────────────────────────────────────────────────────────────────────
  [PASS] TransposeConv(res=False) shape (32,32) == (32,32)
  [PASS] TransposeConv(res=False) SO(3) equivariance              rel=1.35e-05  tol=1e-04  (mean=5.56e-06)
  [PASS] TransposeConv(res=False) crystal-group equivariance (24 ops)  rel=9.71e-06  tol=1e-04  (mean=3.79e-06)
  [PASS] TransposeConv(res=True) shape (32,32) == (32,32)
  [PASS] TransposeConv(res=True) SO(3) equivariance               rel=4.98e-07  tol=1e-04  (mean=2.30e-07)
  [PASS] TransposeConv(res=True) crystal-group equivariance (24 ops)  rel=3.66e-07  tol=1e-04  (mean=1.66e-07)

  [PASS]  6/6 tests passed



In [9]:
# ── Left crystal-group invariance with ENCODER features ──────────────────────

section('TransposeConv — Left crystal-group invariance on encoder features')

q_tc = rand_quaternions(H * W, SEED, DEVICE)
with torch.no_grad():
    x_enc = enc.forward_a1(q_tc).unsqueeze(0)   # (1, H*W, C)

inv_results = []
for res in (False, True):
    torch.manual_seed(SEED + int(res))
    tc = EquivariantTransposeConv(
        kernel_size=3, upsample_factor=UPSAMPLE,
        use_residual=res, irreps_in=irr_a1, irreps_out=irr_a1
    ).to(DEVICE).eval()
    fn = lambda x, _tc=tc: _tc(x, (H, W))[0]

    mean_e, max_e, _ = feature_left_invariance_errors(fn, x_enc, sym_ops, irr_a1, n_ops)
    ok = report(
        f'TransposeConv(res={res}) left-inv on enc features: f(D(s)·feat)=f(feat)',
        max_e, TOL, extra=f'mean={mean_e:.2e}  over {n_ops} ops',
    )
    inv_results.append(ok)

summary(inv_results)



──────────────────────────────────────────────────────────────────────
  TransposeConv — Left crystal-group invariance on encoder features
──────────────────────────────────────────────────────────────────────
  [FAIL] TransposeConv(res=False) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=1.44e+00  tol=1e-04  (mean=1.33e+00  over 24 ops)
  [FAIL] TransposeConv(res=True) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=1.45e+00  tol=1e-04  (mean=1.32e+00  over 24 ops)

  [FAIL]  0/2 tests passed



---
## 4. AttentionBlock — SO(3) equivariance + attention weight invariance

| Test | Expected | Reason |
|---|---|---|
| SO(3) delta-equivariance | ✅ PASS | Attention weights are SO(3)-invariant (next row), e3nn ops are equivariant |
| Attention weight invariance | ✅ PASS | `D` orthogonal → `(Df̂_i)·(Df̂_j) = f̂_i·f̂_j` |
| Crystal-group equivariance | ✅ PASS | O ⊂ SO(3) |

In [10]:
results = []
section('AttentionBlock')

bh = bw = BLOCK_SIZE
ys = torch.linspace(-1,1,bh,device=DEVICE)
xs = torch.linspace(-1,1,bw,device=DEVICE)
gy, gx = torch.meshgrid(ys, xs, indexing='ij')
coords  = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)
d_block = torch.cdist(coords, coords, p=2)

torch.manual_seed(SEED)
x_a1 = torch.randn(1, H*W, irr_a1.dim, dtype=torch.float32, device=DEVICE)

for num_ch in (4, 8):
    torch.manual_seed(SEED + num_ch)
    blk = AttentionBlock(irreps_feat=irr_a1, num_channels=num_ch).to(DEVICE).eval()
    fn  = lambda x, _b=blk: _b(x, d_block, H, W, bh, bw)

    # SO(3) delta equivariance
    mean_e, max_e = so3_equivariance_errors(fn, x_a1, irr_a1, irr_a1, N_TRIALS, SEED)
    results.append(report(f'AttentionBlock(ch={num_ch}) SO(3) delta-equivariance',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

    # Attention weight invariance
    mean_ai, max_ai = attn_weight_invariance_errors(blk, x_a1, irr_a1, d_block, N_TRIALS, SEED)
    results.append(report(f'AttentionBlock(ch={num_ch}) attn-weight SO(3)-invariance',
                          max_ai, 1e-5, extra=f'mean={mean_ai:.2e}'))

    # Crystal-group equivariance
    mean_e, max_e, _ = crystal_so3_equivariance_errors(fn, x_a1, sym_ops, irr_a1, n_ops)
    results.append(report(f'AttentionBlock(ch={num_ch}) crystal-group equivariance ({n_ops} ops)',
                          max_e, TOL, extra=f'mean={mean_e:.2e}'))

summary(results)


──────────────────────────────────────────────────────────────────────
  AttentionBlock
──────────────────────────────────────────────────────────────────────
  [PASS] AttentionBlock(ch=4) SO(3) delta-equivariance            rel=0.00e+00  tol=1e-04  (mean=0.00e+00)
  [PASS] AttentionBlock(ch=4) attn-weight SO(3)-invariance        rel=1.13e-06  tol=1e-05  (mean=6.94e-07)
  [PASS] AttentionBlock(ch=4) crystal-group equivariance (24 ops)  rel=0.00e+00  tol=1e-04  (mean=0.00e+00)
  [PASS] AttentionBlock(ch=8) SO(3) delta-equivariance            rel=0.00e+00  tol=1e-04  (mean=0.00e+00)
  [PASS] AttentionBlock(ch=8) attn-weight SO(3)-invariance        rel=1.13e-06  tol=1e-05  (mean=6.94e-07)
  [PASS] AttentionBlock(ch=8) crystal-group equivariance (24 ops)  rel=0.00e+00  tol=1e-04  (mean=0.00e+00)

  [PASS]  6/6 tests passed



In [11]:
# ── Left crystal-group invariance with ENCODER features ──────────────────────
# d_block, bh, bw defined in the cell above.

section('AttentionBlock — Left crystal-group invariance on encoder features')

q_attn = rand_quaternions(H * W, SEED, DEVICE)
with torch.no_grad():
    x_enc = enc.forward_a1(q_attn).unsqueeze(0)   # (1, H*W, C)

inv_results = []
for num_ch in (4, 8):
    torch.manual_seed(SEED + num_ch)
    blk = AttentionBlock(irreps_feat=irr_a1, num_channels=num_ch).to(DEVICE).eval()
    fn  = lambda x, _b=blk: _b(x, d_block, H, W, bh, bw)

    mean_e, max_e, _ = feature_left_invariance_errors(fn, x_enc, sym_ops, irr_a1, n_ops)
    ok = report(
        f'AttentionBlock(ch={num_ch}) left-inv on enc features: f(D(s)·feat)=f(feat)',
        max_e, TOL, extra=f'mean={mean_e:.2e}  over {n_ops} ops',
    )
    inv_results.append(ok)

summary(inv_results)



──────────────────────────────────────────────────────────────────────
  AttentionBlock — Left crystal-group invariance on encoder features
──────────────────────────────────────────────────────────────────────
  [PASS] AttentionBlock(ch=4) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=0.00e+00  tol=1e-04  (mean=0.00e+00  over 24 ops)
  [PASS] AttentionBlock(ch=8) left-inv on enc features: f(D(s)·feat)=f(feat)  rel=0.00e+00  tol=1e-04  (mean=0.00e+00  over 24 ops)

  [PASS]  2/2 tests passed



---
## 5. Summary

| Layer | Left crystal inv | Right crystal equiv | SO(3) equiv (features) | Attn weight inv |
|---|---|---|---|---|
| Encoder | ✅ enc(s⊗q)=enc(q) | ✅ enc(q⊗s)=enc(q)@D(s) | ❌ N/A | — |
| SpatialConv | — | — | ✅ f(D·x)=D·f(x) | — |
| TransposeConv | — | — | ✅ f(D·x)=D·f(x) | — |
| AttentionBlock | — | — | ✅ delta equivariant | ✅ weights invariant |